# Rossmann Store Sales - Data Understanding (CRISP-DM)

This notebook follows the **Data Understanding** phase of CRISP-DM methodology for time series data.

## Objectives
1. **Initial Data Collection**: Load and inspect raw data
2. **Data Description**: Understand structure, types, and basic statistics
3. **Data Exploration**: Visual and statistical exploration
4. **Data Quality Assessment**: Identify missing values, outliers, and issues

## Key Time Series Considerations
- Temporal patterns (trend, seasonality, cycles)
- Stationarity assessment
- Autocorrelation analysis
- Missing data patterns
- Outlier detection in temporal context

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Output path for figures
OUTPUT_PATH = "/Users/i588313/Library/CloudStorage/OneDrive-SAPSE/SAP/Praxisarbeiten/BA/Typst/assets/data-understanding"
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"Figures will be saved to: {OUTPUT_PATH}")

## 1. Initial Data Collection

### 1.1 Load Datasets

In [ ]:
# Load train data
train = pd.read_csv(
    "../data/train.csv",
    parse_dates=["Date"],
    low_memory=False
)

# Load store metadata
store = pd.read_csv("../data/store.csv")

# Load test data
test = pd.read_csv(
    "../data/test.csv",
    parse_dates=["Date"],
    low_memory=False
)

print(f"Train shape: {train.shape}")
print(f"Store shape: {store.shape}")
print(f"Test shape: {test.shape}")

### 1.2 First Look at Data

In [ ]:
print("=" * 80)
print("TRAIN DATA - First 10 rows")
print("=" * 80)
display(train.head(10))

In [ ]:
print("=" * 80)
print("STORE DATA - First 10 rows")
print("=" * 80)
display(store.head(10))

## 2. Data Description

### 2.1 Dataset Structure and Types

In [ ]:
print("=" * 80)
print("TRAIN DATA INFO")
print("=" * 80)
print(train.info())
print("\n" + "=" * 80)
print("STORE DATA INFO")
print("=" * 80)
print(store.info())

### 2.2 Temporal Coverage

In [ ]:
print("=" * 80)
print("TEMPORAL COVERAGE")
print("=" * 80)
print(f"\nTrain Data:")
print(f"  Start Date: {train['Date'].min()}")
print(f"  End Date: {train['Date'].max()}")
print(f"  Duration: {(train['Date'].max() - train['Date'].min()).days} days")
print(f"  Unique Dates: {train['Date'].nunique()}")

print(f"\nTest Data:")
print(f"  Start Date: {test['Date'].min()}")
print(f"  End Date: {test['Date'].max()}")
print(f"  Duration: {(test['Date'].max() - test['Date'].min()).days} days")
print(f"  Unique Dates: {test['Date'].nunique()}")

print(f"\nGap between train and test: {(test['Date'].min() - train['Date'].max()).days} days")

### 2.3 Store Coverage

In [ ]:
print("=" * 80)
print("STORE COVERAGE")
print("=" * 80)
print(f"\nUnique Stores in Train: {train['Store'].nunique()}")
print(f"Unique Stores in Store metadata: {store['Store'].nunique()}")
print(f"Unique Stores in Test: {test['Store'].nunique()}")

# Check if all stores in train are in store metadata
missing_stores = set(train['Store'].unique()) - set(store['Store'].unique())
print(f"\nStores in train but not in store metadata: {len(missing_stores)}")
if missing_stores:
    print(f"Missing stores: {sorted(missing_stores)}")

### 2.4 Descriptive Statistics

In [ ]:
print("=" * 80)
print("TRAIN DATA - NUMERICAL STATISTICS")
print("=" * 80)
display(train.describe())

In [ ]:
print("=" * 80)
print("TRAIN DATA - CATEGORICAL STATISTICS")
print("=" * 80)
categorical_cols = train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col}:")
    print(train[col].value_counts())
    print(f"Unique values: {train[col].nunique()}")

In [ ]:
print("=" * 80)
print("STORE DATA - CATEGORICAL STATISTICS")
print("=" * 80)
store_categorical = store.select_dtypes(include=['object']).columns
for col in store_categorical:
    print(f"\n{col}:")
    print(store[col].value_counts())
    print(f"Unique values: {store[col].nunique()}")

## 3. Data Quality Assessment

### 3.1 Missing Values Analysis

In [ ]:
def missing_values_table(df, name):
    """
    Generate missing values analysis table
    """
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * mis_val / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table_ren_columns = mis_val_table.rename(
        columns={0: 'Missing Values', 1: '% of Total Values'})
    mis_val_table_ren_columns = mis_val_table_ren_columns[
        mis_val_table_ren_columns.iloc[:, 1] != 0].sort_values(
        '% of Total Values', ascending=False).round(2)
    
    print(f"\n{'='*80}")
    print(f"MISSING VALUES - {name}")
    print(f"{'='*80}")
    print(f"Total columns: {df.shape[1]}")
    print(f"Columns with missing values: {mis_val_table_ren_columns.shape[0]}")
    
    if mis_val_table_ren_columns.shape[0] == 0:
        print("No missing values found!")
    else:
        display(mis_val_table_ren_columns)
    
    return mis_val_table_ren_columns

train_missing = missing_values_table(train, "TRAIN DATA")
store_missing = missing_values_table(store, "STORE DATA")
test_missing = missing_values_table(test, "TEST DATA")

### 3.2 Missing Data Patterns in Time

In [ ]:
# Check if missing values are concentrated in specific time periods
if 'Open' in train.columns:
    train_with_date = train.copy()
    train_with_date['Year'] = train_with_date['Date'].dt.year
    train_with_date['Month'] = train_with_date['Date'].dt.month
    
    missing_by_time = train_with_date.groupby(['Year', 'Month']).apply(
        lambda x: x.isnull().sum()
    )
    
    print("\nMissing values by Year-Month (showing columns with missing data):")
    cols_with_missing = [col for col in missing_by_time.columns if missing_by_time[col].sum() > 0]
    if cols_with_missing:
        display(missing_by_time[cols_with_missing])
    else:
        print("No missing values in train data!")

### 3.3 Zero Sales Analysis

In [ ]:
print("=" * 80)
print("ZERO SALES ANALYSIS")
print("=" * 80)

zero_sales = train[train['Sales'] == 0]
print(f"\nTotal records: {len(train):,}")
print(f"Records with zero sales: {len(zero_sales):,} ({100*len(zero_sales)/len(train):.2f}%)")

# Relationship between zero sales and store being closed
if 'Open' in train.columns:
    zero_sales_open = train[(train['Sales'] == 0) & (train['Open'] == 1)]
    zero_sales_closed = train[(train['Sales'] == 0) & (train['Open'] == 0)]
    
    print(f"\nZero sales when store OPEN: {len(zero_sales_open):,} ({100*len(zero_sales_open)/len(zero_sales):.2f}% of zero sales)")
    print(f"Zero sales when store CLOSED: {len(zero_sales_closed):,} ({100*len(zero_sales_closed)/len(zero_sales):.2f}% of zero sales)")

### 3.4 Outlier Detection - Sales

In [ ]:
# Exclude zero sales for outlier analysis
sales_nonzero = train[train['Sales'] > 0]['Sales']

# IQR method
Q1 = sales_nonzero.quantile(0.25)
Q3 = sales_nonzero.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = sales_nonzero[(sales_nonzero < lower_bound) | (sales_nonzero > upper_bound)]

print("=" * 80)
print("OUTLIER ANALYSIS - SALES (IQR Method)")
print("=" * 80)
print(f"\nQ1 (25th percentile): {Q1:,.0f}")
print(f"Q3 (75th percentile): {Q3:,.0f}")
print(f"IQR: {IQR:,.0f}")
print(f"Lower bound: {lower_bound:,.0f}")
print(f"Upper bound: {upper_bound:,.0f}")
print(f"\nNumber of outliers: {len(outliers):,} ({100*len(outliers)/len(sales_nonzero):.2f}% of non-zero sales)")
print(f"Min outlier: {outliers.min():,.0f}")
print(f"Max outlier: {outliers.max():,.0f}")

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(sales_nonzero, bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(upper_bound, color='r', linestyle='--', linewidth=2, label=f'Upper Bound: {upper_bound:,.0f}')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Non-Zero Sales')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(sales_nonzero, vert=True)
axes[1].set_ylabel('Sales')
axes[1].set_title('Box Plot of Non-Zero Sales')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/01_sales_distribution_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Exploratory Data Analysis

### 4.1 Target Variable Analysis

In [ ]:
# Visualization 4.1a: Sales Distribution Analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sales distribution (excluding zeros)
sales_nonzero = train[train['Sales'] > 0]['Sales']
axes[0].hist(sales_nonzero, bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sales (excluding zeros)')
axes[0].grid(True, alpha=0.3)

# Log-transformed sales
axes[1].hist(np.log1p(sales_nonzero), bins=100, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Log(Sales + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Log-Transformed Sales')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/02a_sales_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Visualization 4.1b: Customer Analysis
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Customers distribution (excluding zeros)
customers_nonzero = train[train['Customers'] > 0]['Customers']
axes[0].hist(customers_nonzero, bins=100, edgecolor='black', alpha=0.7, color='green')
axes[0].set_xlabel('Customers')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Customers (excluding zeros)')
axes[0].grid(True, alpha=0.3)

# Sales vs Customers correlation
sample_data = train[train['Sales'] > 0].sample(min(10000, len(train)))
axes[1].scatter(sample_data['Customers'], sample_data['Sales'], alpha=0.3, s=10)
axes[1].set_xlabel('Customers')
axes[1].set_ylabel('Sales')
axes[1].set_title('Sales vs Customers (sample of 10k points)')
axes[1].grid(True, alpha=0.3)

# Calculate and display correlation
corr = train[train['Sales'] > 0][['Sales', 'Customers']].corr().iloc[0, 1]
axes[1].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
             transform=axes[1].transAxes, 
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/02b_customer_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.2 Temporal Patterns

#### 4.2.1 Overall Time Series

In [ ]:
# Aggregate daily sales across all stores
daily_sales = train.groupby('Date').agg({
    'Sales': 'sum',
    'Customers': 'sum',
}).reset_index()

daily_sales["Open_Stores"] = (train[train["Open"] == 1].groupby("Date").size()).values
daily_sales.columns = ['Date', 'Total_Sales', 'Total_Customers', 'Open_Stores']

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Total daily sales
axes[0].plot(daily_sales['Date'], daily_sales['Total_Sales'], linewidth=0.8)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Total Sales')
axes[0].set_title('Total Daily Sales Across All Stores')
axes[0].grid(True, alpha=0.3)

# Total daily customers
axes[1].plot(daily_sales['Date'], daily_sales['Total_Customers'], linewidth=0.8, color='green')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Total Customers')
axes[1].set_title('Total Daily Customers Across All Stores')
axes[1].grid(True, alpha=0.3)

# Number of open stores
axes[2].plot(daily_sales['Date'], daily_sales['Open_Stores'], linewidth=0.8, color='orange')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Number of Open Stores')
axes[2].set_title('Number of Open Stores per Day')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/03_daily_time_series.png', dpi=300, bbox_inches='tight')
plt.show()

#### 4.2.2 Trend Analysis

In [ ]:
# Monthly aggregation
train_monthly = train.copy()
train_monthly['YearMonth'] = train_monthly['Date'].dt.to_period('M')

monthly_sales = train_monthly.groupby('YearMonth').agg({
    'Sales': 'sum',
    'Customers': 'sum'
}).reset_index()

monthly_sales['YearMonth'] = monthly_sales['YearMonth'].dt.to_timestamp()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Monthly sales trend
axes[0].plot(monthly_sales['YearMonth'], monthly_sales['Sales'], marker='o', linewidth=2)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Total Sales')
axes[0].set_title('Monthly Sales Trend')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Monthly customers trend
axes[1].plot(monthly_sales['YearMonth'], monthly_sales['Customers'], marker='o', linewidth=2, color='green')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Customers')
axes[1].set_title('Monthly Customers Trend')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/04_monthly_trends.png', dpi=300, bbox_inches='tight')
plt.show()

#### 4.2.3 Seasonality Analysis

In [ ]:
# Day of week pattern
dow_sales = train[train['Sales'] > 0].groupby('DayOfWeek').agg({
    'Sales': ['mean', 'median', 'std'],
    'Customers': ['mean', 'median']
}).reset_index()

dow_sales.columns = ['DayOfWeek', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Day of week - Sales
axes[0].bar(dow_sales['DayOfWeek'], dow_sales['Sales_Mean'], 
            yerr=dow_sales['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('Day of Week (1=Mon, 7=Sun)')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Day of Week')
axes[0].set_xticks(range(1, 8))
axes[0].grid(True, alpha=0.3, axis='y')

# Day of week - Customers
axes[1].bar(dow_sales['DayOfWeek'], dow_sales['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('Day of Week (1=Mon, 7=Sun)')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Day of Week')
axes[1].set_xticks(range(1, 8))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/05_weekly_seasonality.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nDay of Week Statistics:")
display(dow_sales)

In [ ]:
# Monthly seasonality
train_with_month = train.copy()
train_with_month['Month'] = train_with_month['Date'].dt.month

monthly_pattern = train_with_month[train_with_month['Sales'] > 0].groupby('Month').agg({
    'Sales': ['mean', 'median', 'std'],
    'Customers': ['mean', 'median']
}).reset_index()

monthly_pattern.columns = ['Month', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Monthly - Sales
axes[0].bar(monthly_pattern['Month'], monthly_pattern['Sales_Mean'], 
            yerr=monthly_pattern['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Month')
axes[0].set_xticks(range(1, 13))
axes[0].grid(True, alpha=0.3, axis='y')

# Monthly - Customers
axes[1].bar(monthly_pattern['Month'], monthly_pattern['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Month')
axes[1].set_xticks(range(1, 13))
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/06_monthly_seasonality.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nMonthly Statistics:")
display(monthly_pattern)

### 4.3 Store Characteristics Analysis

In [ ]:
# Merge train with store metadata
train_store = train.merge(store, on='Store', how='left')

# Sales by store type
store_type_sales = train_store[train_store['Sales'] > 0].groupby('StoreType').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

store_type_sales.columns = ['StoreType', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Store Type - Sales
axes[0].bar(store_type_sales['StoreType'], store_type_sales['Sales_Mean'], 
            yerr=store_type_sales['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('Store Type')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Store Type')
axes[0].grid(True, alpha=0.3, axis='y')

# Store Type - Customers
axes[1].bar(store_type_sales['StoreType'], store_type_sales['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('Store Type')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Store Type')
axes[1].grid(True, alpha=0.3, axis='y')

# Store Type - Count
axes[2].bar(store_type_sales['StoreType'], store_type_sales['Count'], 
            alpha=0.7, color='orange')
axes[2].set_xlabel('Store Type')
axes[2].set_ylabel('Number of Records')
axes[2].set_title('Number of Records by Store Type')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/07_store_type_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nStore Type Statistics:")
display(store_type_sales)

In [ ]:
# Sales by assortment
assortment_sales = train_store[train_store['Sales'] > 0].groupby('Assortment').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

assortment_sales.columns = ['Assortment', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Assortment - Sales
axes[0].bar(assortment_sales['Assortment'], assortment_sales['Sales_Mean'], 
            yerr=assortment_sales['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('Assortment')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Assortment Type')
axes[0].grid(True, alpha=0.3, axis='y')

# Assortment - Customers
axes[1].bar(assortment_sales['Assortment'], assortment_sales['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('Assortment')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Assortment Type')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/07a_store_assortment_type_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nAssortment Statistics:")
display(assortment_sales)

### 4.4 Promotional Analysis

In [ ]:
# Promo effect
promo_sales = train[train['Sales'] > 0].groupby('Promo').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

promo_sales.columns = ['Promo', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Promo - Sales
axes[0].bar(promo_sales['Promo'], promo_sales['Sales_Mean'], 
            yerr=promo_sales['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('Promo (0=No, 1=Yes)')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Promo Status')
axes[0].set_xticks([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Calculate lift
promo_lift = (promo_sales[promo_sales['Promo']==1]['Sales_Mean'].values[0] / 
              promo_sales[promo_sales['Promo']==0]['Sales_Mean'].values[0] - 1) * 100
axes[0].text(0.5, 0.95, f'Promo Lift: {promo_lift:.1f}%', 
             transform=axes[0].transAxes, 
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Promo - Customers
axes[1].bar(promo_sales['Promo'], promo_sales['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('Promo (0=No, 1=Yes)')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Promo Status')
axes[1].set_xticks([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

customer_lift = (promo_sales[promo_sales['Promo']==1]['Customers_Mean'].values[0] / 
                 promo_sales[promo_sales['Promo']==0]['Customers_Mean'].values[0] - 1) * 100
axes[1].text(0.5, 0.95, f'Customer Lift: {customer_lift:.1f}%', 
             transform=axes[1].transAxes, 
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/08_promo_effects.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPromo Statistics:")
display(promo_sales)

In [ ]:
# Promo2 effect
promo2_sales = train_store[train_store['Sales'] > 0].groupby('Promo2').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

promo2_sales.columns = ['Promo2', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Promo2 - Sales
axes[0].bar(promo2_sales['Promo2'], promo2_sales['Sales_Mean'], 
            yerr=promo2_sales['Sales_Std'], capsize=5, alpha=0.7, color='purple')
axes[0].set_xlabel('Promo2 (0=No, 1=Yes)')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by Promo2 Status')
axes[0].set_xticks([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Promo2 - Customers
axes[1].bar(promo2_sales['Promo2'], promo2_sales['Customers_Mean'], 
            alpha=0.7, color='orange')
axes[1].set_xlabel('Promo2 (0=No, 1=Yes)')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by Promo2 Status')
axes[1].set_xticks([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nPromo2 Statistics:")
display(promo2_sales)

### 4.5 Holiday Effects

In [ ]:
# State Holiday effect
state_holiday_sales = train[train['Sales'] > 0].groupby('StateHoliday').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

state_holiday_sales.columns = ['StateHoliday', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

# Mapping für lesbare Labels
holiday_labels = {
    '0': 'No holiday',
    'a': 'Public holiday',
    'b': 'Easter holiday',
    'c': 'Christmas',
    0: 'No holiday'
}
state_holiday_sales['HolidayLabel'] = state_holiday_sales['StateHoliday'].map(holiday_labels)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# State Holiday - Sales
axes[0].bar(range(len(state_holiday_sales)), state_holiday_sales['Sales_Mean'], 
            yerr=state_holiday_sales['Sales_Std'], capsize=5, alpha=0.7)
#axes[0].set_xlabel('State Holiday')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by State Holiday Type')
axes[0].set_xticks(range(len(state_holiday_sales)))
axes[0].set_xticklabels(state_holiday_sales['HolidayLabel'], rotation=15, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

# State Holiday - Customers
axes[1].bar(range(len(state_holiday_sales)), state_holiday_sales['Customers_Mean'], 
            alpha=0.7, color='green')
#axes[1].set_xlabel('State Holiday')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by State Holiday Type')
axes[1].set_xticks(range(len(state_holiday_sales)))
axes[1].set_xticklabels(state_holiday_sales['HolidayLabel'], rotation=15, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/08a_state_holiday_effects.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nState Holiday Statistics:")
display(state_holiday_sales[['StateHoliday', 'HolidayLabel', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']])

In [ ]:
# School Holiday effect
school_holiday_sales = train[train['Sales'] > 0].groupby('SchoolHoliday').agg({
    'Sales': ['mean', 'median', 'std', 'count'],
    'Customers': ['mean', 'median']
}).reset_index()

school_holiday_sales.columns = ['SchoolHoliday', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count', 'Customers_Mean', 'Customers_Median']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# School Holiday - Sales
axes[0].bar(school_holiday_sales['SchoolHoliday'], school_holiday_sales['Sales_Mean'], 
            yerr=school_holiday_sales['Sales_Std'], capsize=5, alpha=0.7)
axes[0].set_xlabel('School Holiday (0=No, 1=Yes)')
axes[0].set_ylabel('Average Sales')
axes[0].set_title('Average Sales by School Holiday Status')
axes[0].set_xticks([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# School Holiday - Customers
axes[1].bar(school_holiday_sales['SchoolHoliday'], school_holiday_sales['Customers_Mean'], 
            alpha=0.7, color='green')
axes[1].set_xlabel('School Holiday (0=No, 1=Yes)')
axes[1].set_ylabel('Average Customers')
axes[1].set_title('Average Customers by School Holiday Status')
axes[1].set_xticks([0, 1])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nSchool Holiday Statistics:")
display(school_holiday_sales)

### 4.6 Competition Analysis

In [ ]:
# Competition distance distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution of competition distance
comp_dist = store['CompetitionDistance'].dropna()
axes[0].hist(comp_dist, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Competition Distance (m)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Competition Distance')
axes[0].grid(True, alpha=0.3)

# Log scale
axes[1].hist(np.log1p(comp_dist), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Log(Competition Distance + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Log Competition Distance')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nCompetition Distance Statistics:")
print(f"Mean: {comp_dist.mean():,.0f} m")
print(f"Median: {comp_dist.median():,.0f} m")
print(f"Min: {comp_dist.min():,.0f} m")
print(f"Max: {comp_dist.max():,.0f} m")
print(f"Missing values: {store['CompetitionDistance'].isnull().sum()} ({100*store['CompetitionDistance'].isnull().sum()/len(store):.1f}%)")

In [ ]:
# Analyze sales vs competition distance
# Bin competition distance
train_store_comp = train_store[train_store['Sales'] > 0].copy()
train_store_comp['CompDistBin'] = pd.cut(
    train_store_comp['CompetitionDistance'], 
    bins=[0, 500, 1000, 2000, 5000, 50000], 
    labels=['<500m', '500-1000m', '1000-2000m', '2000-5000m', '>5000m']
)

comp_dist_sales = train_store_comp.groupby('CompDistBin').agg({
    'Sales': ['mean', 'median', 'std', 'count']
}).reset_index()

comp_dist_sales.columns = ['CompDistBin', 'Sales_Mean', 'Sales_Median', 'Sales_Std', 'Count']

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(comp_dist_sales)), comp_dist_sales['Sales_Mean'], 
       yerr=comp_dist_sales['Sales_Std'], capsize=5, alpha=0.7)
ax.set_xlabel('Competition Distance')
ax.set_ylabel('Average Sales')
ax.set_title('Average Sales by Competition Distance')
ax.set_xticks(range(len(comp_dist_sales)))
ax.set_xticklabels(comp_dist_sales['CompDistBin'])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nSales by Competition Distance:")
display(comp_dist_sales)

### 4.7 Correlation Analysis

In [ ]:
# Select numerical features for correlation
# Merge mit store für CompetitionDistance
train_corr = train[train['Sales'] > 0].merge(store[['Store', 'CompetitionDistance']], on='Store', how='left')

# StateHoliday numerisch kodieren (1 wenn irgendein Feiertag, 0 sonst)
train_corr['StateHoliday'] = (train_corr['StateHoliday'] != '0').astype(int)

numerical_features = ['Sales', 'Customers', 'Promo', 'SchoolHoliday', 'StateHoliday', 'CompetitionDistance', 'DayOfWeek']
train_corr_subset = train_corr[numerical_features].copy()

# Calculate correlation matrix
corr_matrix = train_corr_subset.corr()

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title('Correlation Matrix (Non-Zero Sales)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/09_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCorrelation with Sales (sorted):")
sales_corr = corr_matrix['Sales'].sort_values(ascending=False)
display(sales_corr)

print("\nNote: StateHoliday = 1 wenn Public/Easter/Christmas holiday, 0 sonst")

In [ ]:
# Correlation with store features
store_features = ['CompetitionDistance', 'Promo2']
train_store_corr = train_store[train_store['Sales'] > 0][['Sales', 'Customers'] + store_features].copy()

corr_matrix_store = train_store_corr.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix_store, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
ax.set_title('Correlation Matrix - With Store Features', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

### 4.8 Store-Level Analysis

In [ ]:
# Analyze individual store performance
store_stats = train[train['Sales'] > 0].groupby('Store').agg({
    'Sales': ['mean', 'std', 'min', 'max', 'count'],
    'Customers': ['mean']
}).reset_index()

store_stats.columns = ['Store', 'Sales_Mean', 'Sales_Std', 'Sales_Min', 'Sales_Max', 'Count', 'Customers_Mean']
store_stats['CV'] = store_stats['Sales_Std'] / store_stats['Sales_Mean']  # Coefficient of Variation

# Top 10 stores by average sales
top10_stores = store_stats.nlargest(10, 'Sales_Mean')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top 10 stores by sales
axes[0].barh(range(len(top10_stores)), top10_stores['Sales_Mean'], alpha=0.7)
axes[0].set_yticks(range(len(top10_stores)))
axes[0].set_yticklabels(top10_stores['Store'])
axes[0].set_xlabel('Average Sales')
axes[0].set_ylabel('Store ID')
axes[0].set_title('Top 10 Stores by Average Sales')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].invert_yaxis()

# Distribution of coefficient of variation
axes[1].hist(store_stats['CV'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Coefficient of Variation (CV = Std/Mean)')
axes[1].set_ylabel('Number of Stores')
axes[1].set_title('Distribution of Sales Variability Across Stores')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(store_stats['CV'].median(), color='r', linestyle='--', linewidth=2, label=f"Median CV: {store_stats['CV'].median():.2f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nStore Performance Statistics:")
print(f"Total unique stores: {len(store_stats)}")
print(f"Average sales per store: {store_stats['Sales_Mean'].mean():,.0f}")
print(f"Median sales per store: {store_stats['Sales_Mean'].median():,.0f}")
print(f"Average CV: {store_stats['CV'].mean():.3f}")
print(f"Median CV: {store_stats['CV'].median():.3f}")

### 4.9 Sample Store Time Series

In [ ]:
# Plot time series for a few representative stores
sample_stores = [1, 2, 5, 10]  # Can be adjusted

fig, axes = plt.subplots(len(sample_stores), 1, figsize=(15, 4*len(sample_stores)))

if len(sample_stores) == 1:
    axes = [axes]

for idx, store_id in enumerate(sample_stores):
    store_data = train[train['Store'] == store_id].sort_values('Date')
    
    axes[idx].plot(store_data['Date'], store_data['Sales'], linewidth=0.8, alpha=0.7)
    axes[idx].set_xlabel('Date')
    axes[idx].set_ylabel('Sales')
    axes[idx].set_title(f'Daily Sales - Store {store_id}')
    axes[idx].grid(True, alpha=0.3)
    
    # Add statistics
    stats_text = f"Mean: {store_data['Sales'].mean():,.0f}\nStd: {store_data['Sales'].std():,.0f}"
    axes[idx].text(0.02, 0.95, stats_text, 
                   transform=axes[idx].transAxes, 
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 5. Data Quality Summary

### 5.1 Key Findings

In [ ]:
print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

print("\n1. TEMPORAL COVERAGE")
print(f"   - Train: {train['Date'].min()} to {train['Date'].max()}")
print(f"   - Duration: {(train['Date'].max() - train['Date'].min()).days} days")
print(f"   - Test: {test['Date'].min()} to {test['Date'].max()}")
print(f"   - Gap: {(test['Date'].min() - train['Date'].max()).days} days")

print("\n2. MISSING VALUES")
if len(train_missing) > 0:
    print("   Train data:")
    for idx, row in train_missing.iterrows():
        print(f"     - {idx}: {row['Missing Values']} ({row['% of Total Values']:.1f}%)")
else:
    print("   - No missing values in train data")

if len(store_missing) > 0:
    print("   Store data:")
    for idx, row in store_missing.iterrows():
        print(f"     - {idx}: {row['Missing Values']} ({row['% of Total Values']:.1f}%)")
else:
    print("   - No missing values in store data")

print("\n3. ZERO SALES")
zero_pct = 100 * len(train[train['Sales'] == 0]) / len(train)
print(f"   - {len(train[train['Sales'] == 0]):,} records ({zero_pct:.2f}%)")

print("\n4. OUTLIERS")
outlier_pct = 100 * len(outliers) / len(sales_nonzero)
print(f"   - {len(outliers):,} outliers ({outlier_pct:.2f}% of non-zero sales)")
print(f"   - Upper bound: {upper_bound:,.0f}")

print("\n5. STORE COVERAGE")
print(f"   - Unique stores: {train['Store'].nunique()}")
print(f"   - All stores have metadata: {len(missing_stores) == 0}")

print("\n6. KEY PATTERNS IDENTIFIED")
print(f"   - Strong weekly seasonality (day of week effect)")
print(f"   - Monthly/yearly seasonality present")
print(f"   - Promo effect: {promo_lift:.1f}% lift in sales")
print(f"   - High correlation between Sales and Customers: {corr:.3f}")
print(f"   - Store heterogeneity: CV ranges from {store_stats['CV'].min():.2f} to {store_stats['CV'].max():.2f}")

### 5.2 Recommendations for Data Preparation

Based on this analysis, here are key recommendations for the data preparation phase:

1. **Handle Zero Sales**: 
   - Distinguish between stores being closed vs. truly zero sales when open
   - Consider filtering or special treatment for closed days

2. **Missing Values**:
   - Fill missing competition data with appropriate strategies
   - Consider domain knowledge for Promo2 interval missing values

3. **Feature Engineering**:
   - Create lag features (previous days, weeks)
   - Create rolling statistics (moving averages)
   - Encode cyclical features (day of week, month) with sine/cosine
   - Create interaction features (Promo × DayOfWeek, etc.)

4. **Store-Level Features**:
   - Consider store-specific scaling or normalization
   - Create features capturing store performance trends

5. **Temporal Features**:
   - Add features for holidays and special events
   - Consider competition opening duration
   - Add features for promotional periods

6. **Outlier Treatment**:
   - Investigate outliers in context (special events, holidays)
   - Consider robust scaling methods

7. **Train/Test Split**:
   - Use time-based validation (not random)
   - Consider gap between train and test for lag features